In [13]:
## 📚 Library Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler
import warnings

# Suppress warnings for cleaner output (e.g., chained assignment warnings)
warnings.filterwarnings('ignore')

# ---
# ## 💾 Data Loading and Initial Inspection

# Define the file path
file_path = 'USD_Economy_Centurion_Real_Data_v2.csv'

# Load the dataset
try:
    df = pd.read_csv(
        file_path, 
        index_col=0,       # Set the first column as the index (dates)
        parse_dates=True    # Ensure the index column is parsed as datetime objects
    )

    # 📊 Print Data Summary
    print("✅ DATA LOADED SUCCESSFULLY!")
    print("\n📊 INITIAL DATASET SHAPE:", df.shape)
    
    # Check if the index is a DatetimeIndex before printing min/max
    if isinstance(df.index, pd.DatetimeIndex):
        print("📅 DATE RANGE:", df.index.min().strftime('%Y-%m-%d'), "to", df.index.max().strftime('%Y-%m-%d'))
    else:
        print("⚠️ Index is not a DatetimeIndex. Check the date format in the CSV.")
    
    print("\n📝 DATA TYPES (First 5 Rows):")
    print(df.head())
    
except FileNotFoundError:
    print(f"❌ ERROR: File not found at path: {file_path}")
    df = None # Set df to None if loading fails

# ---

✅ DATA LOADED SUCCESSFULLY!

📊 INITIAL DATASET SHAPE: (9456, 102)
📅 DATE RANGE: 2000-01-01 to 2025-11-20

📝 DATA TYPES (First 5 Rows):
            US_Fed_Funds_Daily  US_Fed_Funds_Target_Up  \
Date                                                     
2000-01-01                3.99                     NaN   
2000-01-02                3.99                     NaN   
2000-01-03                5.43                     NaN   
2000-01-04                5.38                     NaN   
2000-01-05                5.41                     NaN   

            US_Interest_On_Reserves  US_Fed_Balance_Sheet  \
Date                                                        
2000-01-01                      NaN                   NaN   
2000-01-02                      NaN                   NaN   
2000-01-03                      NaN                   NaN   
2000-01-04                      NaN                   NaN   
2000-01-05                      NaN                   NaN   

            US_Treasuries_Fed_

In [15]:
def analyze_and_preserve_early_nan(df: pd.DataFrame):
    """
    Analyzes which time series columns start later than the dataset's beginning
    and prints a report.
    """
    print("="*50)
    print("⏰ STEP 4 — ANALYZE & PRESERVE EARLY MISSING VALUES")
    print("="*50)

    # Get the earliest date in the index
    start_date = df.index.min()

    # Find the first valid date for *each* column
    first_valid_dates = df.apply(lambda col: col.first_valid_index())

    print(f"Dataset officially begins on: **{start_date.strftime('%Y-%m-%d')}**\n")
    print("Reporting indicators that start *after* the dataset beginning:")

    late_start_count = 0
    
    # Iterate and print only the columns that start late
    for col in df.columns:
        first_date = first_valid_dates[col]
        
        # Check if the first valid date is NOT the start date
        if pd.notna(first_date) and first_date > start_date:
            days_late = (first_date - start_date).days
            
            # Print with clear formatting
            print(f"  ➡️ **{col:<30}**: starts {first_date.strftime('%Y-%m-%d')} (Lag: +{days_late} days)")
            late_start_count += 1
            
    print("\n" + "="*50)
    if late_start_count == 0:
        print("✅ **PERFECT START**: All indicators have data on the very first day.")
    else:
        print(f"📈 **SUMMARY**: {late_start_count} indicators start later than the dataset beginning.")
        print("⭐ **Action**: These early NaN values are now documented and **preserved**. They should typically be left alone as they reflect when the data series officially began (not an error).")
    print("="*50)

# Execute the analysis function
analyze_and_preserve_early_nan(df)

⏰ STEP 4 — ANALYZE & PRESERVE EARLY MISSING VALUES
Dataset officially begins on: **2000-01-01**

Reporting indicators that start *after* the dataset beginning:
  ➡️ **US_Fed_Funds_Target_Up        **: starts 2008-12-16 (Lag: +3272 days)
  ➡️ **US_Interest_On_Reserves       **: starts 2021-07-29 (Lag: +7880 days)
  ➡️ **US_Fed_Balance_Sheet          **: starts 2002-12-18 (Lag: +1082 days)
  ➡️ **US_Treasuries_Fed_Held        **: starts 2002-12-18 (Lag: +1082 days)
  ➡️ **US_Bank_Credit                **: starts 2000-01-05 (Lag: +4 days)
  ➡️ **US_Financial_Stress           **: starts 2000-01-07 (Lag: +6 days)
  ➡️ **US_Chicago_Fin_Cond           **: starts 2000-01-07 (Lag: +6 days)
  ➡️ **US_TED_Spread                 **: starts 2000-01-04 (Lag: +3 days)
  ➡️ **US_Yield_30Y                  **: starts 2000-01-03 (Lag: +2 days)
  ➡️ **US_Yield_20Y                  **: starts 2000-01-03 (Lag: +2 days)
  ➡️ **US_Yield_10Y                  **: starts 2000-01-03 (Lag: +2 days)
  ➡️ **US_Yiel

In [16]:
import pandas as pd # Ensure pandas is imported if running this block alone

def impute_and_flag_nan(df: pd.DataFrame):
    """
    Handles remaining NaN values by creating multiple imputed versions
    (0.0, mean, median, mask) and adding NaN flag columns for each original
    column that contained missing data.
    
    Args:
        df (pd.DataFrame): The input DataFrame that may contain NaNs.

    Returns:
        tuple: A tuple containing the four imputed and flagged DataFrames.
    """
    print("="*50)
    print("🤖 STEP 6 — HANDLE NaN WITH CONSTANT REPLACEMENT & FLAGGING")
    print("="*50)

    # Calculate total remaining NaN values
    remaining_nan = df.isna().sum().sum()
    print(f"Total remaining NaN values across all columns: **{remaining_nan}**")

    # Initialize all resulting DataFrames to the original in case of no NaNs
    df_zero_with_flags = df.copy()
    df_mean_with_flags = df.copy()
    df_median_with_flags = df.copy()
    df_dl_with_flags = df.copy()
    nan_flags = pd.DataFrame(index=df.index)

    if remaining_nan > 0:
        
        # 1. IDENTIFY NaN PATTERN & CREATE FLAG MATRIX
        # Identify columns with NaNs
        nan_by_column = df.isna().sum()
        nan_columns = nan_by_column[nan_by_column > 0]
        
        # Create NaN flag matrix BEFORE filling (contains 1 where NaN was, 0 otherwise)
        nan_flags = df[nan_columns.index].isna().astype(int)
        nan_flags = nan_flags.add_suffix('_NAN_FLAG')
        
        print(f"\n🚩 Creating {nan_flags.shape[1]} NaN flag columns...")
        print("📊 NaN Distribution by Column:")
        
        # Report the distribution of NaNs
        for col, count in nan_columns.items():
            percentage = (count / len(df)) * 100
            print(f"  ➡️ {col:<30}: {count} NaN ({percentage:.2f}%)")
            
        print("\n--- Imputation Strategies ---")
        
        # 2. PERFORM IMPUTATION AND CONCATENATION
        
        # Strategy A: Tree-based Models (often best to use the original DF if allowed)
        print("✅ Strategy A: **Original DF** preserved (ideal for XGBoost/LightGBM)")
        
        # Strategy B: Replace with Constant (0.0) + Flags
        print("\n🔄 Strategy B: Replace with **0.0** (Suitable when NaN implies 'zero activity')")
        df_zero = df.fillna(0.0)
        df_zero_with_flags = pd.concat([df_zero, nan_flags], axis=1)
        
        # Strategy C: Replace with Mean + Flags (for linear models/feature engineering)
        print("\n🔄 Strategy C: Replace with **Mean**")
        df_mean = df.fillna(df.mean())
        df_mean_with_flags = pd.concat([df_mean, nan_flags], axis=1)
        
        # Strategy D: Replace with Median + Flags (more robust to outliers than mean)
        print("\n🔄 Strategy D: Replace with **Median**")
        df_median = df.fillna(df.median())
        df_median_with_flags = pd.concat([df_median, nan_flags], axis=1)
        
        # Strategy E: Deep Learning/Masking (-9999) + Flags (clear signal for the model)
        print("\n🔄 Strategy E: Replace with **-9999 Mask** (for Neural Networks/Deep Learning)")
        df_dl_mask = df.fillna(-9999)
        df_dl_with_flags = pd.concat([df_dl_mask, nan_flags], axis=1)
        
        print("\n--- Summary ---")
        print(f"✅ Created multiple imputed versions, each with {nan_flags.shape[1]} flag columns:")
        print(f"  - **df_zero_with_flags**: {df_zero_with_flags.shape} (NaN → 0.0)")
        print(f"  - **df_mean_with_flags**: {df_mean_with_flags.shape} (NaN → mean)")
        print(f"  - **df_median_with_flags**: {df_median_with_flags.shape} (NaN → median)")
        print(f"  - **df_dl_with_flags**: {df_dl_with_flags.shape} (NaN → -9999)")
        
        # 3. VALIDATION AND EXAMPLE
        print(f"\n✅ VALIDATION: Post-imputation NaN count is 0:")
        print(f"  - Zero version NaN: {df_zero.isna().sum().sum()}")
        print(f"  - Mean version NaN: {df_mean.isna().sum().sum()}")
        
        example_col = nan_columns.index[0] # Take the first column with NaNs
        flag_col = f"{example_col}_NAN_FLAG"
        
        # Find the first index where the original column was NaN
        nan_index = nan_flags[nan_flags[flag_col] == 1].index[0]

        print(f"\n🔍 EXAMPLE of Flagging on column '{example_col}' at date {nan_index.strftime('%Y-%m-%d')}:")
        print(f"  - **Flag Value**: {df_zero_with_flags.loc[nan_index, flag_col]}")
        print(f"  - **Original Value**: {df.loc[nan_index, example_col]}")
        print(f"  - **Imputed (Zero) Value**: {df_zero_with_flags.loc[nan_index, example_col]}")

    else:
        print("🎉 No remaining NaN values! Skipping imputation.")
        
    print("="*50)

    # Return the versions created
    return df_zero_with_flags, df_mean_with_flags, df_median_with_flags, df_dl_with_flags


# Example Usage (assuming 'df' is the DataFrame from previous steps)
# df_zero_f, df_mean_f, df_median_f, df_dl_f = impute_and_flag_nan(df)

In [17]:
import pandas as pd
import numpy as np

# Assuming 'df' is the DataFrame from the previous steps
# The analysis below uses the same logic as STEP 4 to identify the columns

def impute_early_nans_with_flag(df: pd.DataFrame) -> pd.DataFrame:
    """
    Identifies early NaN periods, replaces them with 0.0, and creates 
    a corresponding binary flag column for each imputed series.
    """
    print("="*50)
    print("🚩 STEP 5 — IMPUTATION OF EARLY NaNs WITH 0.0 & FLAGGING")
    print("="*50)
    
    # 1. Setup
    df_imputed = df.copy()
    start_date = df.index.min()
    first_valid_dates = df.apply(lambda col: col.first_valid_index())
    
    imputed_count = 0
    flag_columns = []

    # 2. Iterate through columns and perform imputation/flagging
    for col in df.columns:
        first_date = first_valid_dates[col]
        
        # Only process columns that start LATER than the dataset start date
        if pd.notna(first_date) and first_date > start_date:
            imputed_count += 1
            
            # --- 2a. Create the Flag Column ---
            flag_col_name = f"{col}_FLAG_EARLY_NAN"
            
            # The flag is 1 for the dates BEFORE the first valid entry, 0 otherwise.
            # We use loc to select the rows up to (but not including) the first valid date.
            # Initialize with 0
            df_imputed[flag_col_name] = 0
            
            # Set flag to 1 for the early NaN period
            df_imputed.loc[:first_date - pd.Timedelta(days=1), flag_col_name] = 1
            flag_columns.append(flag_col_name)

            # --- 2b. Perform Imputation ---
            # Impute NaNs with 0.0 ONLY in the period identified by the flag (i.e., before first_date)
            # This ensures we don't accidentally fill NaNs that occur *later* in the series.
            
            # Select the block of early NaNs
            early_nan_block = df_imputed.loc[:first_date - pd.Timedelta(days=1), col]
            
            # Replace NaNs in that block with 0.0 (and only NaNs!)
            df_imputed.loc[early_nan_block.index, col] = early_nan_block.fillna(0.0)

            print(f"  ✅ Imputed **{col}** with 0.0 from {start_date.strftime('%Y-%m-%d')} to {(first_date - pd.Timedelta(days=1)).strftime('%Y-%m-%d')}.")
            print(f"     Flag column **{flag_col_name}** created.")

    # 3. Final Summary
    print("\n" + "="*50)
    if imputed_count == 0:
        print("✅ **PERFECT START**: No early NaNs needed imputation or flagging.")
    else:
        print(f"⭐ **SUMMARY**: {imputed_count} series were imputed with 0.0 and flagged.")
        print(f"📊 New DataFrame shape: {df_imputed.shape} (Added {len(flag_columns)} columns)")
    print("="*50)
    
    return df_imputed

# Run the function
df = impute_early_nans_with_flag(df)

🚩 STEP 5 — IMPUTATION OF EARLY NaNs WITH 0.0 & FLAGGING
  ✅ Imputed **US_Fed_Funds_Target_Up** with 0.0 from 2000-01-01 to 2008-12-15.
     Flag column **US_Fed_Funds_Target_Up_FLAG_EARLY_NAN** created.
  ✅ Imputed **US_Interest_On_Reserves** with 0.0 from 2000-01-01 to 2021-07-28.
     Flag column **US_Interest_On_Reserves_FLAG_EARLY_NAN** created.
  ✅ Imputed **US_Fed_Balance_Sheet** with 0.0 from 2000-01-01 to 2002-12-17.
     Flag column **US_Fed_Balance_Sheet_FLAG_EARLY_NAN** created.
  ✅ Imputed **US_Treasuries_Fed_Held** with 0.0 from 2000-01-01 to 2002-12-17.
     Flag column **US_Treasuries_Fed_Held_FLAG_EARLY_NAN** created.
  ✅ Imputed **US_Bank_Credit** with 0.0 from 2000-01-01 to 2000-01-04.
     Flag column **US_Bank_Credit_FLAG_EARLY_NAN** created.
  ✅ Imputed **US_Financial_Stress** with 0.0 from 2000-01-01 to 2000-01-06.
     Flag column **US_Financial_Stress_FLAG_EARLY_NAN** created.
  ✅ Imputed **US_Chicago_Fin_Cond** with 0.0 from 2000-01-01 to 2000-01-06.
     Flag 

In [18]:
import pandas as pd
import numpy as np

# Assuming 'df' is the DataFrame resulting from the previous steps (with numeric types enforced)

def impute_early_nans_with_flag(df: pd.DataFrame) -> pd.DataFrame:
    """
    Identifies periods where the data series had not started (early NaNs), 
    replaces them with 0.0, and creates a corresponding binary flag column.
    """
    print("="*50)
    print("🚩 STEP 5 — IMPUTATION OF EARLY NaNs WITH 0.0 & FLAGGING")
    print("="*50)
    
    # Use a copy to avoid modifying the original DataFrame unintentionally
    df_imputed = df.copy()
    start_date = df_imputed.index.min()
    
    # Identify the first non-NaN date for every column
    first_valid_dates = df_imputed.apply(lambda col: col.first_valid_index())
    
    imputed_count = 0
    flag_columns = []

    # Iterate through columns
    for col in df.columns:
        first_date = first_valid_dates[col]
        
        # Condition: Only proceed if the series starts strictly after the dataset start date
        if pd.notna(first_date) and first_date > start_date:
            imputed_count += 1
            
            # --- 1. Define the Imputation Period (Masking) ---
            # Create a Boolean Mask to select the indices *before* the first valid date.
            # This is the 'early NaN' period we want to change.
            imputation_mask = df_imputed.index < first_date 
            
            # --- 2. Create the Flag Column (Flagging) ---
            flag_col_name = f"{col}_FLAG_EARLY_NAN"
            df_imputed[flag_col_name] = np.where(imputation_mask, 1, 0)
            flag_columns.append(flag_col_name)

            # --- 3. Perform Imputation ---
            # Replace NaNs in the specified period (using the mask) with 0.0
            # We use .loc to ensure the change is made directly to the copy of the DataFrame.
            df_imputed.loc[imputation_mask, col] = df_imputed.loc[imputation_mask, col].fillna(0.0)
            
            # --- 4. Reporting ---
            end_date_imputed = df_imputed.loc[imputation_mask].index.max()
            
            print(f"  ✅ Imputed **{col:<30}** with 0.0 from {start_date.strftime('%Y-%m-%d')} to {end_date_imputed.strftime('%Y-%m-%d')}")

    # Final Summary
    print("\n" + "="*50)
    if imputed_count == 0:
        print("✅ **No Action Required**: No indicators started later than the dataset beginning.")
    else:
        print(f"⭐ **SUMMARY**: {imputed_count} series were imputed with 0.0 and flagged.")
        print(f"📊 New DataFrame shape: {df_imputed.shape} ({len(flag_columns)} new flag columns added)")
    print("="*50)
    
    return df_imputed

# Execute the function and update the main DataFrame
df = impute_early_nans_with_flag(df)

🚩 STEP 5 — IMPUTATION OF EARLY NaNs WITH 0.0 & FLAGGING

✅ **No Action Required**: No indicators started later than the dataset beginning.


In [19]:
import pandas as pd
import numpy as np

# Assuming 'df' is the DataFrame resulting from the previous steps (STEP 5)

def forward_fill_mid_series_nans(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies forward-filling (ffill) to all columns to handle mid-series NaNs,
    assuming the last known value persists.
    """
    print("="*50)
    print("⏩ STEP 6 — IMPUTATION VIA FORWARD FILLING (FFILL)")
    print("="*50)
    
    # We use a copy for safe operations and comparison
    df_filled = df.copy()

    # Calculate the number of NaNs BEFORE imputation for reporting
    nans_before = df_filled.isnull().sum().sum()
    
    # --- 1. Perform Forward Filling ---
    # Apply ffill across the entire DataFrame. 
    # NOTE: This will NOT fill the very first NaNs in columns that started late 
    # (like US_Interest_On_Reserves, which we filled with 0.0 in STEP 5).
    df_filled = df_filled.fillna(method='ffill')

    # Calculate the number of NaNs AFTER imputation
    nans_after = df_filled.isnull().sum().sum()
    
    # --- 2. Calculate Remaining NaNs (Optional Final Check) ---
    # After ffill, the ONLY remaining NaNs should be at the very end of the series
    # (if the last recorded value was NaN) or in columns that were entirely NaN,
    # or the early NaNs if you hadn't run STEP 5 (which you did).
    
    # The sum of NaNs remaining in the core indicator columns
    # We exclude the newly created FLAG columns from this check.
    indicator_columns = [col for col in df_filled.columns if '_FLAG_EARLY_NAN' not in col]
    nans_remaining_indicators = df_filled[indicator_columns].isnull().sum().sum()


    # --- 3. Reporting and Validation ---
    
    print(f"Total NaNs before FFILL: {nans_before:,.0f}")
    
    # Calculate the number of NaNs filled by ffill
    nans_filled = nans_before - nans_after
    
    print(f"Total NaNs filled by FFILL: {nans_filled:,.0f}")
    
    print(f"Total NaNs remaining in ALL columns: {nans_after:,.0f}")

    print("\n" + "="*50)
    if nans_filled > 0:
        print("✅ **SUCCESS**: Mid-series NaNs successfully imputed using the last observed value.")
    else:
        print("✅ **SUCCESS**: No mid-series NaNs were found or needed imputation.")
        
    if nans_remaining_indicators > 0:
        print(f"⚠️ **ACTION REQUIRED**: {nans_remaining_indicators:,.0f} NaNs remain in indicator columns.")
        print("   These are likely at the **very end** of certain series (no new data available).")
        print("   Consider **backward filling (`bfill`)** or simple **interpolation** next.")
    else:
        print("⭐ **CLEAN**: All indicator series are now complete (except for any late-starting periods filled with 0.0).")
    
    print("="*50)
    
    return df_filled

# Execute the function and update the main DataFrame
df = forward_fill_mid_series_nans(df)

⏩ STEP 6 — IMPUTATION VIA FORWARD FILLING (FFILL)
Total NaNs before FFILL: 722,182
Total NaNs filled by FFILL: 722,182
Total NaNs remaining in ALL columns: 0

✅ **SUCCESS**: Mid-series NaNs successfully imputed using the last observed value.
⭐ **CLEAN**: All indicator series are now complete (except for any late-starting periods filled with 0.0).


In [20]:
## 🌲 Outlier Detection (Multivariate using Isolation Forest)

from sklearn.ensemble import IsolationForest

def detect_isolation_forest_outliers(df: pd.DataFrame, contamination_rate: float = 0.01) -> pd.DataFrame:
    """
    Uses Isolation Forest to detect global outliers based on all features.
    Creates a new column '_OUTLIER_IFOREST' with 1 (Outlier) or 0 (Normal).
    
    Args:
        df: The DataFrame containing the features.
        contamination_rate: The expected proportion of outliers in the data (e.g., 0.01 = 1%).
        
    Returns:
        The DataFrame with the outlier flag column added.
    """
    print("="*50)
    print("🌲 STEP 7.1 — MULTIVARIATE OUTLIER DETECTION (Isolation Forest)")
    print("="*50)

    df_outlier = df.copy()

    # Exclude Flag columns, as they are binary indicators and can skew distance-based methods
    feature_cols = [col for col in df_outlier.columns if '_FLAG_' not in col]
    X = df_outlier[feature_cols].dropna() # Drop any remaining NaNs for the model

    if X.empty:
        print("⚠️ DataFrame is empty or contains too many NaNs for Isolation Forest. Skipping.")
        return df

    # Initialize and train the Isolation Forest model
    # random_state ensures reproducibility
    model = IsolationForest(
        contamination=contamination_rate, 
        random_state=42, 
        n_jobs=-1
    )
    model.fit(X)

    # Predict the outliers: -1 for outlier, 1 for inlier
    # The output is mapped to 1 (outlier) or 0 (normal)
    outlier_predictions = model.predict(X)
    
    # Create the flag column, initially 0 (Normal) for all rows
    df_outlier['_OUTLIER_IFOREST'] = 0
    
    # Map the model's predictions (-1 -> 1, 1 -> 0) back to the original index
    outlier_indices = X.index[outlier_predictions == -1]
    df_outlier.loc[outlier_indices, '_OUTLIER_IFOREST'] = 1
    
    # Reporting
    outlier_count = df_outlier['_OUTLIER_IFOREST'].sum()
    print(f"🌲 Isolation Forest Outliers Detected:")
    print(f"   - **{outlier_count}** observations flagged as multivariate outliers (approx. {outlier_count / len(df_outlier):.2%}).")
    print(f"   - **First 5 outlier dates:** {list(outlier_indices[:5].strftime('%Y-%m-%d'))}")
    
    return df_outlier

# Execute the Isolation Forest detection
df = detect_isolation_forest_outliers(df, contamination_rate=0.005) # Adjust rate as needed

🌲 STEP 7.1 — MULTIVARIATE OUTLIER DETECTION (Isolation Forest)
🌲 Isolation Forest Outliers Detected:
   - **47** observations flagged as multivariate outliers (approx. 0.50%).
   - **First 5 outlier dates:** ['2020-04-01', '2020-04-02', '2020-04-03', '2020-04-04', '2020-04-05']


In [22]:
def review_multivariate_outliers(df: pd.DataFrame):
    """
    Reports the dates flagged by the Isolation Forest model.
    """
    print("="*50)
    print("📝 STEP 8.2 — REVIEW MULTIVARIATE OUTLIER DATES")
    print("="*50)
    
    # Check if the flag column exists (from STEP 7.1)
    if '_OUTLIER_IFOREST' not in df.columns:
        print("⚠️ Outlier flag column '_OUTLIER_IFOREST' not found. Skip review.")
        return

    # Filter the DataFrame to show only the flagged outliers
    outlier_df = df[df['_OUTLIER_IFOREST'] == 1]
    outlier_count = len(outlier_df)
    
    if outlier_count == 0:
        print("✅ No observations were flagged as multivariate outliers.")
        return

    print(f"🌲 Isolation Forest Flagged **{outlier_count}** Multivariate Outliers.")
    print("These are time periods where the joint behavior of all 102 indicators was highly abnormal.")
    
    # Identify unique years/months to see where the outliers cluster
    outlier_dates = outlier_df.index
    date_clusters = outlier_dates.to_period('M').value_counts().sort_index()

    print("\n📅 **Outlier Cluster Analysis (Count by Month/Year):**")
    
    if not date_clusters.empty:
        for period, count in date_clusters.items():
            # Highlight clusters with more than a few days flagged
            if count > 2:
                print(f"   ➡️ **{period}**: {count} days flagged")
            else:
                 print(f"   - {period}: {count} day(s) flagged")

        print("\n⭐ **Observation**: The cluster around **2020-04** is highly consistent with global economic disruption.")
        print("   Such outliers are likely genuine events, not data errors, and should often be preserved or handled with robust scaling.")
    
    print("="*50)

# Execute the review
review_multivariate_outliers(df)

📝 STEP 8.2 — REVIEW MULTIVARIATE OUTLIER DATES
🌲 Isolation Forest Flagged **47** Multivariate Outliers.
These are time periods where the joint behavior of all 102 indicators was highly abnormal.

📅 **Outlier Cluster Analysis (Count by Month/Year):**
   ➡️ **2020-04**: 30 days flagged
   ➡️ **2020-05**: 17 days flagged

⭐ **Observation**: The cluster around **2020-04** is highly consistent with global economic disruption.
   Such outliers are likely genuine events, not data errors, and should often be preserved or handled with robust scaling.


In [23]:
## 📏 Feature Scaling with RobustScaler

from sklearn.preprocessing import RobustScaler
import pandas as pd

# Assuming 'df' is the DataFrame resulting from the previous steps

def perform_robust_scaling(df: pd.DataFrame) -> tuple[pd.DataFrame, RobustScaler]:
    """
    Applies Robust Scaling to all primary indicator columns (excluding flags) 
    and returns the scaled DataFrame and the fitted scaler object.
    """
    print("="*50)
    print("⚖️ STEP 9 — ROBUST FEATURE SCALING")
    print("="*50)
    
    # 1. Identify Columns to Scale
    # Exclude all binary flag columns (they should not be scaled)
    cols_to_scale = [col for col in df.columns if '_FLAG_' not in col and col != '_OUTLIER_IFOREST']
    
    # Isolate the data to be scaled
    X_raw = df[cols_to_scale]
    
    # 2. Initialize and Fit Scaler
    # Use RobustScaler because the data contains confirmed, genuine outliers (2020 events)
    scaler = RobustScaler()
    print("Initializing RobustScaler (using Median and IQR for robustness)...")
    
    # Fit the scaler using the non-NaN values
    # We must drop NaNs here for the fit, although in this stage, there should be none left in the indicators
    X_scaled_array = scaler.fit_transform(X_raw) 
    
    # 3. Create the Scaled DataFrame
    df_scaled = pd.DataFrame(X_scaled_array, columns=cols_to_scale, index=df.index)
    
    # 4. Re-attach Flag and Outlier Columns
    # We add the unscaled binary flags back to the scaled DataFrame
    flag_cols = [col for col in df.columns if '_FLAG_' in col or col == '_OUTLIER_IFOREST']
    df_scaled[flag_cols] = df[flag_cols]
    
    print("\n✅ Scaling Complete.")
    print(f"   - **{len(cols_to_scale)}** indicator columns scaled.")
    print(f"   - **{len(flag_cols)}** flag columns preserved unscaled.")
    
    # Display statistics for a sample column after scaling
    sample_col = cols_to_scale[0]
    print("\n📊 Statistics for a Scaled Column (e.g., %s):" % sample_col)
    print(f"   - Median: {df_scaled[sample_col].median():.4f} (Should be close to 0)")
    print(f"   - IQR: {df_scaled[sample_col].quantile(0.75) - df_scaled[sample_col].quantile(0.25):.4f} (Should be close to 1)")
    print(f"   - Min/Max Range: {df_scaled[sample_col].min():.4f} to {df_scaled[sample_col].max():.4f}")
    
    print("="*50)

    return df_scaled, scaler

# Execute the scaling and update the main DataFrame
df_final, robust_scaler = perform_robust_scaling(df)

# You can now use df_final for your model training

⚖️ STEP 9 — ROBUST FEATURE SCALING
Initializing RobustScaler (using Median and IQR for robustness)...

✅ Scaling Complete.
   - **102** indicator columns scaled.
   - **39** flag columns preserved unscaled.

📊 Statistics for a Scaled Column (e.g., US_Fed_Funds_Daily):
   - Median: 0.0000 (Should be close to 0)
   - IQR: 1.0000 (Should be close to 1)
   - Min/Max Range: -0.3169 to 1.4987


In [24]:
import pandas as pd
import numpy as np

# Assuming 'df' is the DataFrame after STEP 7.1

def final_data_integrity_check(df: pd.DataFrame) -> pd.DataFrame:
    """
    Checks for duplicate indices and enforces final numeric data type consistency.
    """
    print("="*50)
    print("✔️ STEP 8.1 — INDEX INTEGRITY & FINAL DATA TYPES")
    print("="*50)
    
    df_clean = df.copy()

    # --- 1. Check for Duplicate Indices ---
    duplicate_indices = df_clean.index.duplicated(keep=False)
    
    if duplicate_indices.any():
        duplicate_count = duplicate_indices.sum()
        print(f"⚠️ **{duplicate_count}** rows found with duplicate dates. Removing duplicates...")
        # Keep the first occurrence, discarding the rest.
        df_clean = df_clean[~df_clean.index.duplicated(keep='first')]
        print(f"✅ Duplicates removed. New shape: {df_clean.shape}")
    else:
        print("✅ No duplicate dates/indices found. Index is unique.")
        
    # --- 2. Enforce Numeric Consistency (Final Check) ---
    non_numeric_before = df_clean.select_dtypes(include=['object']).columns
    
    if len(non_numeric_before) > 0:
        print(f"\n⚠️ {len(non_numeric_before)} columns are currently non-numeric. Enforcing...")
        
        # Convert all columns to float, turning non-convertible values into NaN
        for col in df_clean.columns:
            # We skip FLAG columns which are typically int/bool
            if '_FLAG_' not in col:
                df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        
        non_numeric_after = df_clean.select_dtypes(include=['object']).columns

        if len(non_numeric_after) == 0:
            print("✅ All non-flag columns successfully converted to numeric (float).")
        else:
            print(f"❌ ERROR: {len(non_numeric_after)} columns still contain unconvertible data. Manual review required.")
    else:
        print("✅ No object type columns found. Data types are consistent.")

    return df_clean

# Execute the final integrity check
df = final_data_integrity_check(df)

✔️ STEP 8.1 — INDEX INTEGRITY & FINAL DATA TYPES
✅ No duplicate dates/indices found. Index is unique.
✅ No object type columns found. Data types are consistent.


In [25]:
## 💾 STEP 10 — Save the Final Cleaned Dataset

import pandas as pd
import os

# Use 'df' as the final DataFrame from previous steps

def save_cleaned_dataset(df_to_save: pd.DataFrame, original_name: str) -> None:
    """
    Saves the processed DataFrame to a new CSV file with a suffix.
    """
    print("="*50)
    print("💾 STEP 10 — SAVING FINAL CLEANED DATASET")
    print("="*50)
    
    # 1. Define the output file name
    # We strip the original .csv and append '_PROCESSED.csv'
    base_name = original_name.replace('.csv', '')
    output_filename = f"{base_name}_PROCESSED.csv"
    
    # 2. Save the DataFrame
    try:
        df_to_save.to_csv(output_filename, index=True, index_label='Date')
        
        print(f"✅ SUCCESSFULLY SAVED CLEANED DATASET.")
        print(f"   - **File:** {output_filename}")
        print(f"   - **Shape:** {df_to_save.shape}")
        print(f"   - **Location:** {os.getcwd()}")
        
    except Exception as e:
        print(f"❌ ERROR saving file: {e}")

    print("="*50)


# Assuming the original file name was 'USD_Economy_Centurion_Real_Data_v2.csv'
original_file = 'USD_Economy_Centurion_Real_Data_v2.csv'

# Execute the save function
save_cleaned_dataset(df, original_file)

💾 STEP 10 — SAVING FINAL CLEANED DATASET
✅ SUCCESSFULLY SAVED CLEANED DATASET.
   - **File:** USD_Economy_Centurion_Real_Data_v2_PROCESSED.csv
   - **Shape:** (9456, 141)
   - **Location:** c:\Users\Zabih Ullah Mujahid\OneDrive\Desktop\US economic
